In [ ]:
# ── Imports and setup ──

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import os

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
OUT = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT, exist_ok=True)

delta_clr = pd.read_csv(os.path.join(OUT, "delta_otu_clr.csv"))
abs_clr   = pd.read_csv(os.path.join(OUT, "preprocessed_otu_clr.csv"), index_col="sample_uid")
taxonomy  = pd.read_csv(os.path.join(OUT, "taxonomy_table.csv"), index_col="OTU_ID")

print(f"delta_clr:  {delta_clr.shape}  | columns[:5]: {delta_clr.columns[:5].tolist()}")
print(f"abs_clr:    {abs_clr.shape}")
print(f"taxonomy:   {taxonomy.shape}")

# Identify OTU columns in delta_clr (exclude metadata columns baked in)
META_COLS_DELTA = ['study', 'subject_id', 'treatment']
delta_meta_cols = [c for c in delta_clr.columns if c in META_COLS_DELTA]
delta_otu_cols  = [c for c in delta_clr.columns if c not in META_COLS_DELTA]
print(f"\ndelta metadata cols: {delta_meta_cols}")
print(f"delta OTU cols count: {len(delta_otu_cols)}")

In [ ]:
# ── Load and align data ──
meta_clean = pd.read_csv(os.path.join(OUT, "merged_metadata_clean.csv"), index_col="Unnamed: 0")
meta_clean.index.name = "sample_uid"

abs_clr = pd.read_csv(os.path.join(OUT, "preprocessed_otu_clr.csv"), index_col="sample_uid")

common = abs_clr.index.intersection(meta_clean.index)
abs_clr    = abs_clr.loc[common]
meta_clean = meta_clean.loc[common]

print(f"Common samples: {len(common)}")
print(f"abs_clr:    {abs_clr.shape}")
print(f"meta_clean: {meta_clean.shape}")
print(f"\ntreatment value counts:\n{meta_clean['treatment'].value_counts()}")
print(f"\nfiber_type value counts:\n{meta_clean['fiber_type'].value_counts()}")

In [ ]:
# ── Fiber group mapping ──

FIBER_GROUP_MAP = {
    # Group 1 — Prebiotic oligosaccharides
    "inulin":                               1,
    "FOS":                                  1,
    "GOS":                                  1,
    "oligofructose":                        1,
    "long_chain_inulin":                    1,
    "short_chain_FOS":                      1,

    # Group 2 — Resistant starch
    "himaize":                              2,
    "potato":                               2,
    "potato_RS4A":                          2,
    "potato_RS4B":                          2,
    "potato_RS4C":                          2,
    "potato_starch":                        2,
    "maize":                                2,
    "tapioca":                              2,
    "starch-entrapped-microspheres-12g":    2,
    "starch-entrapped-microspheres-9g":     2,

    # Group 3 — Soluble fiber
    "soluble_corn":                         3,
    "polydextrose":                         3,

    # Group 4 — Grain/wheat fiber
    "barley-kernel-bread":                  4,
    "white-wheat-bread":                    4,

    # Group 5 — Mixed/unspecified
    "fiber_diet_10g":                       5,
    "fiber_diet_40g":                       5,

    # Control
    "starch_control":                       0,
    "maltodextrin_control":                 0,
    "corn_control":                         0,
    "none":                                 0,
}

GROUP_NAMES = {
    0: "Control",
    1: "Prebiotic oligosaccharides",
    2: "Resistant starch",
    3: "Soluble fiber",
    4: "Grain/wheat fiber",
    5: "Mixed/unspecified",
}

# Apply mapping
meta_clean["fiber_group"] = meta_clean["fiber_type"].map(FIBER_GROUP_MAP)

# Report
print("Fiber group distribution:")
for g, name in GROUP_NAMES.items():
    n = (meta_clean["fiber_group"] == g).sum()
    print(f"  Group {g} ({name}): {n} samples")

unmapped = meta_clean["fiber_group"].isna()
print(f"\nUnmapped (excluded from DA): {unmapped.sum()}")
print(meta_clean.loc[unmapped, "fiber_type"].value_counts())

In [ ]:
# ── Study-matched control pools ──
# For each fiber group, controls = control samples from studies that also contain that fiber group

# Work only with mapped samples
meta_mapped = meta_clean[meta_clean["fiber_group"].notna()].copy()
meta_mapped["fiber_group"] = meta_mapped["fiber_group"].astype(int)

# Studies present in each fiber group
group_studies = {}
for g in range(1, 6):
    studies = meta_mapped.loc[meta_mapped["fiber_group"] == g, "study"].unique()
    group_studies[g] = set(studies)
    print(f"Group {g} ({GROUP_NAMES[g]}): {len(studies)} studies — {sorted(studies)}")

# Control samples
all_controls = meta_mapped[meta_mapped["fiber_group"] == 0]
print(f"\nTotal control samples: {len(all_controls)}")
print(f"Control studies:\n{all_controls['study'].value_counts()}")

# Build study-matched control index per group
control_idx = {}
for g in range(1, 6):
    matched = all_controls[all_controls["study"].isin(group_studies[g])]
    control_idx[g] = matched.index
    print(f"\nGroup {g} matched controls: {len(matched)} samples from {matched['study'].nunique()} studies")

In [ ]:
# ── DA groups (Groups 1–3 only) ──
# Group 4 (Kovatcheva): crossover between two fiber arms, no true control
# Group 5 (Tap): no control arm
# DA performed for Groups 1, 2, 3 only

DA_GROUPS = [1, 2, 3]

# Build fiber sample indices per DA group
fiber_idx = {}
for g in DA_GROUPS:
    fiber_idx[g] = meta_mapped[meta_mapped["fiber_group"] == g].index
    print(f"Group {g} ({GROUP_NAMES[g]}): {len(fiber_idx[g])} fiber samples, {len(control_idx[g])} matched controls")

# Align abs_clr to mapped samples only
abs_clr_mapped = abs_clr.loc[meta_mapped.index]
print(f"\nabs_clr_mapped: {abs_clr_mapped.shape}")
otu_cols = abs_clr_mapped.columns.tolist()
print(f"OTU columns: {len(otu_cols)}")


In [ ]:
# ── Mann–Whitney U differential abundance ──
# ~29K tests total, expect 2-5 min runtime

results = []

for g in DA_GROUPS:
    print(f"Running Group {g} ({GROUP_NAMES[g]})...", flush=True)

    fiber_clr   = abs_clr_mapped.loc[fiber_idx[g]].values      # shape: (n_fiber, 9612)
    control_clr = abs_clr_mapped.loc[control_idx[g]].values    # shape: (n_control, 9612)

    pvals, clr_diffs = [], []

    for i in range(len(otu_cols)):
        f = fiber_clr[:, i]
        c = control_clr[:, i]
        stat, p = mannwhitneyu(f, c, alternative="two-sided")
        pvals.append(p)
        clr_diffs.append(f.mean() - c.mean())

    # BH FDR correction
    _, padj, _, _ = multipletests(pvals, method="fdr_bh")

    for i, otu in enumerate(otu_cols):
        results.append({
            "OTU_ID":       otu,
            "fiber_group":  g,
            "group_name":   GROUP_NAMES[g],
            "n_fiber":      len(fiber_idx[g]),
            "n_control":    len(control_idx[g]),
            "clr_diff":     clr_diffs[i],
            "pval":         pvals[i],
            "padj":         padj[i],
            "significant":  (padj[i] < 0.05) and (abs(clr_diffs[i]) > 1.5),
        })

    n_sig = sum(1 for r in results if r["fiber_group"] == g and r["significant"])
    print(f"  Done. Significant OTUs (FDR<0.05, |CLR diff|>1.5): {n_sig}")

da_results = pd.DataFrame(results)
print(f"\nTotal DA results shape: {da_results.shape}")

In [ ]:
# ── Taxonomy annotation and save ──

# Taxonomy: join on OTU_ID, extract genus-level label from GTDB string
# GTDB format: d__; p__; c__; o__; f__; g__; s__
# Extract last informative rank (prefer genus, fall back to family, order etc.)

def extract_label(tax_string):
    if not isinstance(tax_string, str):
        return "Unknown"
    parts = [x.strip() for x in tax_string.split(";")]
    # Walk from species back to domain, return first non-empty non-"Unknown" rank
    for part in reversed(parts):
        rank_val = part.split("__")[-1].strip()
        if rank_val and rank_val.lower() not in ("", "unknown"):
            return part  # return full rank__name string e.g. g__Bifidobacterium
    return "Unknown"

taxonomy["tax_label"] = taxonomy["taxonomy"].apply(extract_label)

# Join taxonomy onto DA results
da_results = da_results.merge(
    taxonomy[["tax_label"]],
    left_on="OTU_ID",
    right_index=True,
    how="left"
)

da_results["tax_label"] = da_results["tax_label"].fillna("Unknown")

# Save full results
da_results.to_csv(os.path.join(OUT, "differential_abundance_results.csv"), index=False)
print(f"Saved differential_abundance_results.csv: {da_results.shape}")

# Summary of significant hits
sig = da_results[da_results["significant"]]
print(f"\nTotal significant OTUs across all groups: {len(sig)}")
print(f"\nBreakdown by group:")
print(sig.groupby(["fiber_group","group_name"])[["OTU_ID"]].count())

print(f"\nTop 10 by |clr_diff| across all groups:")
print(sig.nlargest(10, "clr_diff")[["OTU_ID","group_name","clr_diff","padj","tax_label"]].to_string())

In [ ]:
# ── Supplementary heatmap (top 20) ──

# --- Supplementary: top 20 per group ---
top20_otus_per_group = []
for g in [1, 2]:
    grp_sig = da_results[(da_results["fiber_group"] == g) & (da_results["significant"])]
    top20 = grp_sig.nlargest(20, "clr_diff")[["OTU_ID","tax_label","clr_diff","fiber_group"]]
    top20_otus_per_group.append(top20)

top20_df = pd.concat(top20_otus_per_group, ignore_index=True)
unique_otus_20 = top20_df["OTU_ID"].unique()

heatmap_data_20 = {}
for g in [1, 2]:
    grp = da_results[da_results["fiber_group"] == g].set_index("OTU_ID")
    heatmap_data_20[GROUP_NAMES[g]] = grp.loc[
        grp.index.intersection(unique_otus_20), "clr_diff"
    ].reindex(unique_otus_20)

heatmap_df_20 = pd.DataFrame(heatmap_data_20, index=unique_otus_20)
heatmap_df_20 = heatmap_df_20.sort_values(GROUP_NAMES[1], ascending=False, na_position="last")

label_map_20 = top20_df.drop_duplicates("OTU_ID").set_index("OTU_ID")["tax_label"]

# Clean labels with OTU hash suffix for duplicates
raw_labels_20 = heatmap_df_20.index.map(
    lambda x: label_map_20.get(x, x)
    .replace("s__","").replace("g__","").replace("f__","").strip()
)
label_counts = {}
for l in raw_labels_20:
    label_counts[l] = label_counts.get(l, 0) + 1

seen2 = {}
dedup_labels_20 = []
for otu, lbl in zip(heatmap_df_20.index, raw_labels_20):
    if label_counts[lbl] > 1:
        dedup_labels_20.append(f"{lbl} [{otu[:6]}]")
    else:
        dedup_labels_20.append(lbl)

heatmap_df_20.index = dedup_labels_20

fig, ax = plt.subplots(figsize=(7, 12))
sns.heatmap(
    heatmap_df_20, ax=ax, cmap="RdBu_r", center=0, vmin=-2, vmax=2,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Mean CLR difference (fiber − control)", "shrink": 0.6},
)
ax.set_title(
    "Top 20 Differential OTUs by Fiber Type Group\n(FDR < 0.05, |CLR diff| > 1.5) — Supplementary",
    fontsize=11, fontweight="bold", pad=12
)
ax.set_xlabel("Fiber Group", fontsize=10)
ax.set_ylabel("Taxon", fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "heatmap_top20_otus_supplementary.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved heatmap_top20_otus_supplementary.png")

In [ ]:
# ── Select balanced OTUs (top 20 positive + negative) ──
# Groups 1 and 2 only (Group 3 zero sig OTUs)
# Dense figure: up to 80 unique OTUs across both groups

top_balanced = []
for g in [1, 2]:
    grp_sig = da_results[(da_results["fiber_group"] == g) & (da_results["padj"] < 0.05)]
    
    # Top 20 positive
    pos = grp_sig[grp_sig["clr_diff"] > 0].nlargest(20, "clr_diff")
    # Top 20 negative
    neg = grp_sig[grp_sig["clr_diff"] < 0].nsmallest(20, "clr_diff")
    
    top_balanced.append(pos)
    top_balanced.append(neg)
    
    print(f"Group {g} ({GROUP_NAMES[g]}): {len(pos)} positive, {len(neg)} negative selected")

top_bal_df = pd.concat(top_balanced, ignore_index=True)
unique_otus_bal = top_bal_df["OTU_ID"].unique()
print(f"\nTotal unique OTUs in balanced heatmap: {len(unique_otus_bal)}")

In [ ]:
# ── Balanced heatmap (main figure) ──

# Build heatmap matrix — all sig OTUs, clr_diff value for each group (NaN if not significant)
label_map_bal = top_bal_df.drop_duplicates("OTU_ID").set_index("OTU_ID")["tax_label"]

heatmap_data_bal = {}
for g in [1, 2]:
    grp = da_results[da_results["fiber_group"] == g].set_index("OTU_ID")
    heatmap_data_bal[GROUP_NAMES[g]] = grp.loc[
        grp.index.intersection(unique_otus_bal), "clr_diff"
    ].reindex(unique_otus_bal)

heatmap_df_bal = pd.DataFrame(heatmap_data_bal, index=unique_otus_bal)

# Sort: positives on top (by Group 1 descending), negatives on bottom
heatmap_df_bal["_sort"] = heatmap_df_bal[GROUP_NAMES[1]].fillna(
    heatmap_df_bal[GROUP_NAMES[2]]
)
heatmap_df_bal = heatmap_df_bal.sort_values("_sort", ascending=False)
heatmap_df_bal = heatmap_df_bal.drop(columns="_sort")

# Build clean deduplicated labels with OTU hash suffix for duplicates
raw_labels_bal = heatmap_df_bal.index.map(
    lambda x: label_map_bal.get(x, x)
    .replace("s__","").replace("g__","").replace("f__","").strip()
)
label_counts_bal = {}
for l in raw_labels_bal:
    label_counts_bal[l] = label_counts_bal.get(l, 0) + 1

dedup_labels_bal = []
for otu, lbl in zip(heatmap_df_bal.index, raw_labels_bal):
    if label_counts_bal[lbl] > 1:
        dedup_labels_bal.append(f"{lbl} [{otu[:6]}]")
    else:
        dedup_labels_bal.append(lbl)

heatmap_df_bal.index = dedup_labels_bal

# Plot
fig, ax = plt.subplots(figsize=(8, 16))

sns.heatmap(
    heatmap_df_bal,
    ax=ax,
    cmap="RdBu_r",
    center=0,
    vmin=-2, vmax=2,
    annot=True, fmt=".2f",
    annot_kws={"size": 7.5},
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Mean CLR difference (fiber − control)", "shrink": 0.4},
)

# Horizontal divider between positive and negative blocks
n_pos = (heatmap_df_bal.iloc[:, 0].fillna(heatmap_df_bal.iloc[:, 1]) > 0).sum()
ax.axhline(y=n_pos, color="black", linewidth=1.5, linestyle="--")

ax.set_title(
    "Differential OTUs by Fiber Type Group\nTop 20 Positive + Top 20 Negative Responders (FDR < 0.05)",
    fontsize=12, fontweight="bold", pad=14
)
ax.set_xlabel("Fiber Group", fontsize=10)
ax.set_ylabel("Taxon", fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7.5)
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUT, "heatmap_top_otus_per_fiber_group.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Saved heatmap_top_otus_per_fiber_group.png")

In [ ]:
# ── Stage 5 summary ──

print("=" * 60)
print("STAGE 5 SUMMARY REPORT")
print("=" * 60)

print("\n--- Study Design ---")
print(f"Total samples entering DA: {len(meta_mapped)}")
print(f"Fiber samples: {(meta_mapped['treatment'] == 'fiber').sum()}")
print(f"Control samples: {(meta_mapped['treatment'] == 'control').sum()}")
print(f"Samples excluded (psyllium, unmapped): {unmapped.sum()}")

print("\n--- Fiber Groups ---")
for g in range(0, 6):
    n = (meta_mapped['fiber_group'] == g).sum()
    print(f"  Group {g} ({GROUP_NAMES.get(g, 'Control')}): {n} samples")

print("\n--- DA Groups Tested ---")
print("  Groups 1, 2, 3 tested (study-matched controls available)")
print("  Group 4 (Kovatcheva): EXCLUDED — no control arm (fiber vs fiber crossover)")
print("  Group 5 (Tap): EXCLUDED — no control arm")

print("\n--- Matched Controls per Group ---")
for g in DA_GROUPS:
    studies = sorted(meta_mapped.loc[meta_mapped['fiber_group'] == g, 'study'].unique())
    ctrl_studies = sorted(meta_mapped.loc[control_idx[g], 'study'].unique())
    print(f"  Group {g} ({GROUP_NAMES[g]}):")
    print(f"    Fiber studies : {studies}")
    print(f"    Control studies: {ctrl_studies}")
    print(f"    n fiber: {len(fiber_idx[g])}, n control: {len(control_idx[g])}")

print("\n--- DA Method ---")
print("  Test: Mann-Whitney U (two-sided)")
print("  Correction: Benjamini-Hochberg FDR")
print("  Significance threshold: FDR < 0.05 AND |CLR diff| > 1.5")
print(f"  Total OTUs tested per group: {len(otu_cols)}")
print(f"  Total tests performed: {len(otu_cols) * len(DA_GROUPS)}")

print("\n--- Significant OTUs ---")
for g in DA_GROUPS:
    grp = da_results[da_results['fiber_group'] == g]
    sig = grp[grp['significant']]
    pos = (sig['clr_diff'] > 0).sum()
    neg = (sig['clr_diff'] < 0).sum()
    top_pos = sig[sig['clr_diff'] > 0].nlargest(1, 'clr_diff')[['tax_label','clr_diff','padj']]
    top_neg = sig[sig['clr_diff'] < 0].nsmallest(1, 'clr_diff')[['tax_label','clr_diff','padj']]
    print(f"\n  Group {g} ({GROUP_NAMES[g]}):")
    print(f"    Total significant: {len(sig)}")
    print(f"    Positive responders: {pos}")
    print(f"    Negative responders: {neg}")
    if len(top_pos):
        row = top_pos.iloc[0]
        print(f"    Top positive: {row['tax_label']} (CLR diff={row['clr_diff']:.3f}, padj={row['padj']:.2e})")
    if len(top_neg):
        row = top_neg.iloc[0]
        print(f"    Top negative: {row['tax_label']} (CLR diff={row['clr_diff']:.3f}, padj={row['padj']:.2e})")

print("\n--- Cross-group Consistency ---")
sig1 = set(da_results[(da_results['fiber_group']==1) & da_results['significant']]['OTU_ID'])
sig2 = set(da_results[(da_results['fiber_group']==2) & da_results['significant']]['OTU_ID'])
shared = sig1 & sig2
print(f"  OTUs significant in both Group 1 and Group 2: {len(shared)}")
print(f"  Group 1 only: {len(sig1 - sig2)}")
print(f"  Group 2 only: {len(sig2 - sig1)}")

print("\n--- Outputs Saved ---")
print(f"  differential_abundance_results.csv")
print(f"  heatmap_top_otus_per_fiber_group.png  (main figure)")
print(f"  heatmap_top20_otus_supplementary.png  (supplementary)")
print("=" * 60)